<a href="https://colab.research.google.com/github/ermonsterking/Binary-Change-Detection-EO-SAR-Image-Pairs/blob/main/Galaxy_Eye.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CELL 1 — Mount Google Drive & Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
!mkdir -p /content/galaxeye
%cd /content/galaxeye

print("✓ Drive mounted")
print("✓ Working directory: /content/galaxeye")


Mounted at /content/drive
/content/galaxeye
✓ Drive mounted
✓ Working directory: /content/galaxeye


# CELL 2 — Install Dependencies

In [ ]:
!pip install -q rasterio albumentations pyyaml scikit-learn tqdm matplotlib

# CELL 3 — Download Dataset from HuggingFace

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download
from pathlib import Path

DATA_DIR = Path("/content/galaxeye/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "train").exists():
    print("Downloading dataset... (this takes ~10 min)")
    snapshot_download(
        repo_id="doron333/change-detection-dataset",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        ignore_patterns=["*.git*"],
    )
    print("✓ Download complete")
else:
    print("✓ Dataset already present")

# Verify structure
!ls -lh /content/galaxeye/data/


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✓ Download complete
total 9.4G
-rw-r--r-- 1 root root 243K May 14 03:16 GalaxEye_Technical_Assignment_v2.pdf
-rw-r--r-- 1 root root   33 May 14 03:16 README.md
-rw-r--r-- 1 root root 187M May 14 03:16 test.zip
-rw-r--r-- 1 root root 8.3G May 14 03:18 train.zip
-rw-r--r-- 1 root root 981M May 14 03:17 val.zip


In [ ]:
# Unzip the datasets
!unzip -q /content/galaxeye/data/train.zip -d /content/galaxeye/data/
!unzip -q /content/galaxeye/data/val.zip -d /content/galaxeye/data/
!unzip -q /content/galaxeye/data/test.zip -d /content/galaxeye/data/

# Verify that folders now exist
!ls -d /content/galaxeye/data/*/

/content/galaxeye/data/test/   /content/galaxeye/data/val/
/content/galaxeye/data/train/


In [ ]:
!rm /content/galaxeye/data/train.zip
!rm /content/galaxeye/data/val.zip
!rm /content/galaxeye/data/test.zip

# CELL 4 — Create config.yaml

In [ ]:
%%writefile config.yaml
# ── Paths ────────────────────────────────────────────────────
data:
  root:          "data/"
  eo_name:       "pre_eo.tif"
  sar_name:      "post_sar.tif"
  mask_name:     "target.tif"
  nodata_eo:     0
  nodata_sar:    0
  eo_variants:   ["pre_eo.tif",  "pre_disaster.tif",  "preeo.tif"]
  sar_variants:  ["post_sar.tif","post_disaster.tif", "postsar.tif"]
  mask_variants: ["target.tif",  "mask.tif",          "label.tif"]

# ── Label remapping ──────────────────────────────────────────
label:
  needs_remap: false
  remap: {0: 0, 1: 0, 2: 1, 3: 1}

# ── Model ────────────────────────────────────────────────────
model:
  name:           "CrossModalChangeNet"
  base_channels:  64
  dropout:        0.1
  in_channels:    4

# ── Training ─────────────────────────────────────────────────
train:
  seed:           42
  patch_size:     256
  batch_size:     16
  num_workers:    2
  num_epochs:     80
  patience:       15
  grad_clip:      1.0
  mixed_precision: true

# ── Optimiser ────────────────────────────────────────────────
optimizer:
  name:           "AdamW"
  lr:             0.0001
  weight_decay:   0.0001

# ── LR Scheduler ─────────────────────────────────────────────
scheduler:
  name:           "CosineAnnealingLR"
  eta_min:        0.000001

# ── Loss ─────────────────────────────────────────────────────
loss:
  name:           "DiceFocal"
  dice_weight:    0.5
  focal_weight:   0.5
  focal_alpha:    0.75
  focal_gamma:    2.0

# ── Augmentation ─────────────────────────────────────────────
augmentation:
  random_crop:     true
  horizontal_flip: 0.5
  vertical_flip:   0.5
  random_rotate90: 0.5
  transpose:       0.3

# ── Output ───────────────────────────────────────────────────
norm_stats_path: "norm_stats.json"
output:
  checkpoint_dir: "checkpoints/"
  results_dir:    "results/"
  best_model:     "checkpoints/best_model.pth"

Writing config.yaml


# CELL 5 — Create dataset.py

In [ ]:
%%writefile dataset.py
import numpy as np
import torch
from torch.utils.data import Dataset
import rasterio
import albumentations as A
from pathlib import Path

def collect_samples(data_root: Path, split: str, *args, **kwargs):
    split_dir = data_root / split
    pre_dir, post_dir, target_dir = split_dir / "pre-event", split_dir / "post-event", split_dir / "target"

    samples = []
    if not pre_dir.exists(): return samples

    file_list = sorted([f.name for f in pre_dir.glob("*.tif")])
    for fname in file_list:
        samples.append({
            "eo": str(pre_dir / fname),
            "sar": str(post_dir / fname),
            "mask": str(target_dir / fname)
        })
    return samples

def normalise(arr, mean, std):
    return ((arr - np.array(mean)) / (np.array(std) + 1e-6)).astype(np.float32)

class EOSARDataset(Dataset):
    def __init__(self, samples, norm_stats, mode="train", patch_size=256, **kwargs):
        self.samples = samples
        self.stats = norm_stats
        self.patch_size = patch_size
        self.transforms = A.Compose([
            A.Resize(patch_size, patch_size),
            A.HorizontalFlip(p=0.5) if mode == "train" else A.NoOp(),
            A.VerticalFlip(p=0.5) if mode == "train" else A.NoOp(),
        ], additional_targets={"sar": "image"})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        with rasterio.open(s["eo"]) as src:
            eo = src.read([1, 2, 3]).transpose(1, 2, 0).astype(np.float32) / 255.0
        with rasterio.open(s["sar"]) as src:
            sar = src.read(1).astype(np.float32)[..., np.newaxis] / 255.0
        with rasterio.open(s["mask"]) as src:
            mask = src.read(1).astype(np.uint8)

        # CRITICAL FIX: Ensure mask is strictly binary (0 or 1)
        # We treat any value >= 1 as change.
        # If there are no-data values (like 255), we map those to 0.
        binary_mask = np.where((mask >= 1) & (mask < 255), 1, 0).astype(np.uint8)

        aug = self.transforms(image=eo, sar=sar, mask=binary_mask)

        eo = normalise(aug["image"], self.stats["eo_mean"], self.stats["eo_std"])
        sar = normalise(aug["sar"], self.stats["sar_mean"], self.stats["sar_std"])

        img = np.concatenate([eo, sar], axis=-1)
        return {
            "image": torch.from_numpy(img).permute(2, 0, 1),
            "mask": torch.from_numpy(aug["mask"]).long(),
            "path": s["mask"]
        }

Writing dataset.py


# CELL 6 — Create model.py

In [ ]:
%%writefile model.py
"""CrossModalChangeNet architecture"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, ic: int, oc: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(True),
            nn.Dropout2d(dropout),
            nn.Conv2d(oc, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(True),
        )
    def forward(self, x): return self.net(x)


class Down(nn.Module):
    def __init__(self, ic: int, oc: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(ic, oc, dropout))
    def forward(self, x): return self.net(x)


class Up(nn.Module):
    def __init__(self, ic: int, skip_c: int, oc: int, dropout: float = 0.1):
        super().__init__()
        self.up   = nn.ConvTranspose2d(ic, ic//2, 2, stride=2)
        self.conv = DoubleConv(ic//2 + skip_c, oc, dropout)
    def forward(self, x, skip):
        x  = self.up(x)
        dh = skip.size(2) - x.size(2); dw = skip.size(3) - x.size(3)
        x  = F.pad(x, [dw//2, dw-dw//2, dh//2, dh-dh//2])
        return self.conv(torch.cat([skip, x], 1))


class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, hidden), nn.ReLU(True),
            nn.Linear(hidden, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).view(x.size(0), -1, 1, 1)


class FusionBlock(nn.Module):
    def __init__(self, eo_c: int, sar_c: int, out_c: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(eo_c + sar_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(True),
        )
        self.se = SEBlock(out_c)
    def forward(self, eo_feat, sar_feat):
        if eo_feat.shape[2:] != sar_feat.shape[2:]:
            sar_feat = F.interpolate(sar_feat, size=eo_feat.shape[2:],
                                     mode="bilinear", align_corners=False)
        fused = torch.cat([eo_feat, sar_feat], dim=1)
        return self.se(self.proj(fused))


class CrossModalChangeNet(nn.Module):
    def __init__(self, base_channels: int = 64, dropout: float = 0.1, in_channels: int = 4):
        super().__init__()
        c = base_channels

        self.eo_enc1 = DoubleConv(3,   c,    dropout)
        self.eo_enc2 = Down(c,    c*2, dropout)
        self.eo_enc3 = Down(c*2,  c*4, dropout)
        self.eo_enc4 = Down(c*4,  c*8, dropout)
        self.eo_bot  = Down(c*8, c*16, dropout)

        self.sar_enc1 = DoubleConv(1,   c//2, dropout)
        self.sar_enc2 = Down(c//2, c,   dropout)
        self.sar_enc3 = Down(c,    c*2, dropout)
        self.sar_enc4 = Down(c*2,  c*4, dropout)
        self.sar_bot  = Down(c*4,  c*8, dropout)

        self.fuse_bot  = FusionBlock(c*16, c*8, c*16)
        self.fuse_enc4 = FusionBlock(c*8,  c*4, c*8)
        self.fuse_enc3 = FusionBlock(c*4,  c*2, c*4)
        self.fuse_enc2 = FusionBlock(c*2,  c,   c*2)
        self.fuse_enc1 = FusionBlock(c,    c//2,c)

        self.dec4 = Up(c*16, c*8, c*8, dropout)
        self.dec3 = Up(c*8,  c*4, c*4, dropout)
        self.dec2 = Up(c*4,  c*2, c*2, dropout)
        self.dec1 = Up(c*2,  c,   c,   dropout)

        self.head = nn.Sequential(
            nn.Conv2d(c, c//2, 3, padding=1), nn.ReLU(True),
            nn.Dropout2d(dropout), nn.Conv2d(c//2, 1, 1),
        )

    def _encode_eo(self, x):
        e1 = self.eo_enc1(x); e2 = self.eo_enc2(e1)
        e3 = self.eo_enc3(e2); e4 = self.eo_enc4(e3)
        b  = self.eo_bot(e4)
        return e1, e2, e3, e4, b

    def _encode_sar(self, x):
        s1 = self.sar_enc1(x); s2 = self.sar_enc2(s1)
        s3 = self.sar_enc3(s2); s4 = self.sar_enc4(s3)
        sb = self.sar_bot(s4)
        return s1, s2, s3, s4, sb

    def forward(self, x):
        eo  = x[:, :3]; sar = x[:, 3:]
        e1, e2, e3, e4, eb = self._encode_eo(eo)
        s1, s2, s3, s4, sb = self._encode_sar(sar)
        fb = self.fuse_bot (eb, sb); f4 = self.fuse_enc4(e4, s4)
        f3 = self.fuse_enc3(e3, s3); f2 = self.fuse_enc2(e2, s2)
        f1 = self.fuse_enc1(e1, s1)
        d = self.dec4(fb, f4); d = self.dec3(d, f3)
        d = self.dec2(d, f2);  d = self.dec1(d, f1)
        return self.head(d)


def build_model(cfg: dict):
    return CrossModalChangeNet(
        base_channels=cfg.get("base_channels", 64),
        dropout=cfg.get("dropout", 0.1),
        in_channels=cfg.get("in_channels", 4),
    )

print("✓ model.py created")


Writing model.py


# CELL 7 — Create losses.py

In [ ]:
%%writefile losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        tgts = targets.view(-1).float()
        intersection = (probs * tgts).sum()
        return 1 - (2. * intersection + self.smooth) / (probs.sum() + tgts.sum() + self.smooth)

class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, focal_weight=0.5, alpha=0.75, gamma=2.0):
        super().__init__()
        self.dw, self.fw = dice_weight, focal_weight
        self.dice = DiceLoss()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        logits = logits.squeeze(1) # [B, 1, H, W] -> [B, H, W]
        # BCE with logit flattening
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction="mean")
        dice = self.dice(logits, targets)
        return (self.dw * dice) + (self.fw * bce)

def build_loss(cfg):
    return CombinedLoss(dice_weight=cfg['dice_weight'], focal_weight=cfg['focal_weight'])

Writing losses.py


# CELL 8 — Create utils.py

In [ ]:
%%writefile utils.py
"""Utilities for training"""

import json, random
from pathlib import Path
import numpy as np
import torch
import rasterio
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, jaccard_score
import shutil

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_norm_stats(samples: list[dict], max_n: int = 500,
                       nodata_eo: float = 0.0, nodata_sar: float = 0.0) -> dict:
    from tqdm.auto import tqdm
    eo_sum  = np.zeros(3, np.float64); eo_sq = np.zeros(3, np.float64)
    eo_cnt  = np.zeros(3, np.float64)
    sar_sum = sar_sq = sar_cnt = 0.0
    subset = random.sample(samples, min(max_n, len(samples)))
    for s in tqdm(subset, desc="Norm stats", leave=False):
        with rasterio.open(s["eo"]) as src:
            eo = src.read([1,2,3]).astype(np.float32) / 255.0
        for c in range(3):
            ch = eo[c].ravel()
            valid = ch[ch > nodata_eo/255.0 + 1e-6]
            eo_sum[c] += valid.sum(); eo_sq[c] += (valid**2).sum()
            eo_cnt[c] += len(valid)
        with rasterio.open(s["sar"]) as src:
            sar = src.read(1).astype(np.float32).ravel() / 255.0
        valid = sar[sar > nodata_sar/255.0 + 1e-6]
        sar_sum += valid.sum(); sar_sq += (valid**2).sum(); sar_cnt += len(valid)
    eo_mean = (eo_sum / eo_cnt).tolist()
    eo_std  = np.sqrt(np.maximum(eo_sq/eo_cnt - (eo_sum/eo_cnt)**2, 0)).tolist()
    sar_mean = [float(sar_sum / sar_cnt)]
    sar_std  = [float(np.sqrt(max(sar_sq/sar_cnt - (sar_sum/sar_cnt)**2, 0)))]
    return {"eo_mean": eo_mean, "eo_std": eo_std,
            "sar_mean": sar_mean, "sar_std": sar_std}


def load_or_compute_stats(path, samples, **kwargs):
    path = Path(path)
    if path.exists():
        stats = json.loads(path.read_text())
        print(f"  Loaded norm stats from {path}")
        return stats
    print("  Computing normalisation statistics...")
    stats = compute_norm_stats(samples, **kwargs)
    path.write_text(json.dumps(stats, indent=2))
    print(f"  Saved norm stats → {path}")
    return stats


def compute_metrics(preds: np.ndarray, targets: np.ndarray) -> dict:
    prec, rec, f1, _ = precision_recall_fscore_support(
        targets, preds, average="binary", zero_division=0)
    iou = jaccard_score(targets, preds, average="binary", zero_division=0)
    cm  = confusion_matrix(targets, preds, labels=[0,1])
    return {"iou": float(iou), "precision": float(prec),
            "recall": float(rec), "f1": float(f1), "cm": cm}


def threshold_search(logits: np.ndarray, targets: np.ndarray,
                     low: float = 0.10, high: float = 0.90, step: float = 0.02):
    probs = 1.0 / (1.0 + np.exp(-logits))
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(low, high+step, step):
        preds = (probs >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            targets, preds, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


def save_best_checkpoint(model, epoch, val_f1, val_iou, threshold, norm_stats, cfg, ckpt_dir):
    # Standard local save
    path = ckpt_dir / f"best_model.pth"
    torch.save({
        "epoch": epoch, "model_state_dict": model.state_dict(),
        "val_f1": val_f1, "val_iou": val_iou,
        "threshold": threshold, "norm_stats": norm_stats, "config": cfg,
    }, path)

    # REPLACED: Use shutil instead of !cp
    drive_path = "/content/drive/MyDrive/galaxeye_results/checkpoints/best_model.pth"

    # Ensure the destination directory exists before copying
    Path(drive_path).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(path), drive_path)

    return path


def analyse_imbalance(samples: list[dict], max_n: int = 500) -> float:
    from tqdm.auto import tqdm
    total_change = total_pixels = 0
    for s in tqdm(samples[:max_n], desc="Imbalance", leave=False):
        with rasterio.open(s["mask"]) as src:
            mask = src.read(1).astype(np.uint8)
        total_change += int((mask == 1).sum())
        total_pixels += mask.size
    ratio = total_change / total_pixels
    print(f"  Change pixels: {total_change:,} / {total_pixels:,} = {ratio*100:.2f}%")
    print(f"  Imbalance: 1:{(1-ratio)/ratio:.1f} (no-change:change)")
    return ratio

print("✓ utils.py created")



Overwriting utils.py


# CELL 9 — Create train.py

In [ ]:
%%writefile train.py
"""Training script"""

import argparse, gc, json, sys
from pathlib import Path
import numpy as np
import torch, yaml
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset import EOSARDataset, collect_samples
from losses  import build_loss
from model   import build_model
from utils   import (seed_everything, load_or_compute_stats, compute_metrics,
                     threshold_search, save_best_checkpoint, analyse_imbalance)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--config", type=str, default="config.yaml")
    p.add_argument("--data_root", type=str, default=None)
    return p.parse_args()


def train_epoch(model, loader, optimizer, criterion, scaler, device, grad_clip):
    model.train()
    total = 0.0
    for batch in tqdm(loader, desc="  Train", leave=False):
        imgs  = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        total += loss.item()
        del imgs, masks, loss
    torch.cuda.empty_cache(); gc.collect()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total = 0.0
    all_logits, all_targets = [], []
    for batch in tqdm(loader, desc="  Eval", leave=False):
        imgs  = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        total += loss.item()
        all_logits.append(logits.squeeze(1).cpu().float().numpy().ravel())
        all_targets.append(masks.cpu().numpy().ravel())
        del imgs, masks, logits, loss
    torch.cuda.empty_cache(); gc.collect()

    all_logits  = np.concatenate(all_logits)
    all_targets = np.concatenate(all_targets)
    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    metrics = compute_metrics(preds, all_targets)
    metrics["loss"] = total / len(loader)
    return metrics, all_logits, all_targets


def main():
    args = parse_args()
    with open(args.config) as f:
        cfg = yaml.safe_load(f)
    if args.data_root:
        cfg["data"]["root"] = args.data_root

    seed_everything(cfg["train"]["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")

    DATA_ROOT = Path(cfg["data"]["root"])
    CKPT_DIR  = Path(cfg["output"]["checkpoint_dir"])
    RES_DIR   = Path(cfg["output"]["results_dir"])
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    RES_DIR.mkdir(parents=True, exist_ok=True)

    def _collect(split):
        return collect_samples(DATA_ROOT, split, cfg["data"]["eo_variants"],
                              cfg["data"]["sar_variants"], cfg["data"]["mask_variants"])

    train_samples = _collect("train")
    val_samples   = _collect("val")
    print(f"Samples — train: {len(train_samples)}  val: {len(val_samples)}")

    print("\nClass imbalance:")
    change_ratio = analyse_imbalance(train_samples)

    norm_stats = load_or_compute_stats(
        cfg["norm_stats_path"], train_samples,
        nodata_eo=cfg["data"]["nodata_eo"], nodata_sar=cfg["data"]["nodata_sar"])

    needs_remap = cfg["label"]["needs_remap"]
    patch_size  = cfg["train"]["patch_size"]

    train_ds = EOSARDataset(train_samples, norm_stats, mode="train",
                           patch_size=patch_size, needs_remap=needs_remap,
                           cfg_aug=cfg.get("augmentation"),
                           nodata_eo=cfg["data"]["nodata_eo"],
                           nodata_sar=cfg["data"]["nodata_sar"])
    val_ds = EOSARDataset(val_samples, norm_stats, mode="val",
                         patch_size=patch_size, needs_remap=needs_remap,
                         nodata_eo=cfg["data"]["nodata_eo"],
                         nodata_sar=cfg["data"]["nodata_sar"])

    train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"],
                             shuffle=True, num_workers=cfg["train"]["num_workers"],
                             pin_memory=True, drop_last=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["train"]["batch_size"],
                           shuffle=False, num_workers=cfg["train"]["num_workers"],
                           pin_memory=True, persistent_workers=True)

    model = build_model(cfg["model"]).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    criterion = build_loss(cfg["loss"])
    print(f"\nModel: {cfg['model']['name']} ({n_params:.2f}M params)")

    optimizer = getattr(torch.optim, cfg["optimizer"]["name"])(
        model.parameters(), lr=cfg["optimizer"]["lr"],
        weight_decay=cfg["optimizer"]["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["train"]["num_epochs"],
        eta_min=cfg["scheduler"]["eta_min"])
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"]["mixed_precision"])

    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_iou": []}
    best_val_f1 = 0.0; best_threshold = 0.5; best_ckpt_path = None
    no_improve = 0; patience = cfg["train"]["patience"]

    hdr = f"{'Ep':>4} | {'TrLoss':>7} | {'VaLoss':>7} | {'F1':>7} | {'IoU':>7} | {'LR':>9}"
    print(f"\n{hdr}"); print("-"*len(hdr))

    for epoch in range(1, cfg["train"]["num_epochs"]+1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion,
                             scaler, device, cfg["train"]["grad_clip"])
        va_m, _, _ = eval_epoch(model, val_loader, criterion, device, best_threshold)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_m["loss"])
        history["val_f1"].append(va_m["f1"])
        history["val_iou"].append(va_m["iou"])

        if va_m["f1"] > best_val_f1:
            best_val_f1 = va_m["f1"]; no_improve = 0
            best_ckpt_path = save_best_checkpoint(
                model, epoch, va_m["f1"], va_m["iou"],
                best_threshold, norm_stats, cfg, CKPT_DIR)
            flag = " ★"
        else:
            no_improve += 1; flag = ""

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"{epoch:>4} | {tr_loss:>7.4f} | {va_m['loss']:>7.4f} | "
              f"{va_m['f1']:>7.4f} | {va_m['iou']:>7.4f} | {lr_now:>9.2e}{flag}")

        if no_improve >= patience:
            print(f"\nEarly stop @ epoch {epoch}")
            break

    print(f"\nBest Val F1: {best_val_f1:.4f}")

    print("\nThreshold search...")
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    _, val_logits, val_targets = eval_epoch(model, val_loader, criterion, device, 0.5)
    best_threshold, best_f1 = threshold_search(val_logits, val_targets)
    print(f"Best threshold: {best_threshold:.2f} (Val F1={best_f1:.4f})")

    best_ckpt_path = save_best_checkpoint(
        model, ckpt["epoch"], best_f1, ckpt["val_iou"],
        best_threshold, norm_stats, cfg, CKPT_DIR)

    hist_path = RES_DIR / "training_history.json"
    hist_path.write_text(json.dumps({**history, "best_val_f1": best_val_f1,
                                     "best_threshold": best_threshold}, indent=2))
    print(f"\nDone. Checkpoint → {best_ckpt_path}")

if __name__ == "__main__":
    main()

print("✓ train.py created")




Overwriting train.py


In [ ]:
!find /content/galaxeye/data -maxdepth 3 -type d

/content/galaxeye/data
/content/galaxeye/data/.cache
/content/galaxeye/data/.cache/huggingface
/content/galaxeye/data/.cache/huggingface/download
/content/galaxeye/data/val
/content/galaxeye/data/val/post-event
/content/galaxeye/data/val/target
/content/galaxeye/data/val/pre-event
/content/galaxeye/data/train
/content/galaxeye/data/train/post-event
/content/galaxeye/data/train/target
/content/galaxeye/data/train/pre-event
/content/galaxeye/data/test
/content/galaxeye/data/test/post-event
/content/galaxeye/data/test/target
/content/galaxeye/data/test/pre-event


# CELL 10 — Run Training

In [ ]:
!python train.py --config config.yaml --data_root /content/galaxeye/data/

✓ model.py created
✓ utils.py created

Device: cpu
Samples — train: 2781  val: 334

Class imbalance:
  Change pixels: 63,602,230 / 524,288,000 = 12.13%
  Imbalance: 1:7.2 (no-change:change)
  Computing normalisation statistics...
  Saved norm stats → norm_stats.json

Model: CrossModalChangeNet (38.57M params)
/content/galaxeye/train.py:132: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"]["mixed_precision"])
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(

  Ep |  TrLoss |  VaLoss |      F1 |     IoU |        LR
--------------------------------------------------------
  Train:   0% 0/173 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is se

# CELL 11 — Create eval.py (for later use)

In [ ]:
%%writefile eval.py
"""Evaluation script"""

import argparse, json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import yaml

from dataset import EOSARDataset, collect_samples
from model import build_model
from utils import compute_metrics, seed_everything


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_path", type=str, required=True)
    p.add_argument("--weights", type=str, required=True)
    p.add_argument("--config", type=str, default="config.yaml")
    p.add_argument("--threshold", type=float, default=None)
    p.add_argument("--batch_size", type=int, default=8)
    return p.parse_args()


def main():
    args = parse_args()
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    model = build_model(cfg["model"]).to(device)
    ckpt = torch.load(args.weights, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    threshold = args.threshold if args.threshold else ckpt["threshold"]
    norm_stats = ckpt.get("norm_stats")
    if not norm_stats:
        norm_stats = json.loads(Path(cfg["norm_stats_path"]).read_text())

    model.eval()
    print(f"Loaded: epoch {ckpt['epoch']} val_f1={ckpt['val_f1']:.4f}")
    print(f"Threshold: {threshold:.2f}")

    split_dir = Path(args.data_path)
    split_name = split_dir.name

    samples = collect_samples(
        split_dir.parent, split_name,
        cfg["data"]["eo_variants"], cfg["data"]["sar_variants"],
        cfg["data"]["mask_variants"])
    print(f"Samples: {len(samples)}")

    ds = EOSARDataset(samples, norm_stats, mode="test",
                     patch_size=cfg["train"]["patch_size"],
                     needs_remap=cfg["label"]["needs_remap"],
                     nodata_eo=cfg["data"]["nodata_eo"],
                     nodata_sar=cfg["data"]["nodata_sar"])
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False,
                       num_workers=2, pin_memory=True)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval {split_name}"):
            imgs = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"]
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1).cpu().float().numpy()
            all_logits.append(logits.ravel())
            all_targets.append(masks.numpy().ravel())

    all_logits = np.concatenate(all_logits)
    all_targets = np.concatenate(all_targets)
    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)

    metrics = compute_metrics(preds, all_targets)
    print(f"\n{'='*54}")
    print(f"  {split_name.upper()}  (threshold={threshold:.2f})")
    print(f"{'='*54}")
    print(f"  IoU       : {metrics['iou']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"{'='*54}")

    out = {
        "split": split_name, "threshold": round(threshold, 4),
        "iou": round(metrics["iou"], 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
        "confusion_matrix": metrics["cm"].tolist(),
    }
    out_dir = Path("results")
    out_dir.mkdir(exist_ok=True)
    mpath = out_dir / f"metrics_{split_name}.json"
    mpath.write_text(json.dumps(out, indent=2))
    print(f"Saved → {mpath}")

if __name__ == "__main__":
    main()

print("✓ eval.py created")




Overwriting eval.py


# CELL 12 — Evaluate on Test Set

In [ ]:
!python eval.py \
    --data_path data/test \
    --weights checkpoints/best_ep*.pth


✓ model.py created
✓ utils.py created
Traceback (most recent call last):
  File "/content/galaxeye/eval.py", line 103, in <module>
    main()
  File "/content/galaxeye/eval.py", line 35, in main
    ckpt = torch.load(args.weights, map_location=device)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/best_ep*.pth'


# CELL 13 — Copy Results to Google Drive (for safekeeping)

In [ ]:
!mkdir -p /content/drive/MyDrive/galaxeye_results
!cp -r checkpoints/ /content/drive/MyDrive/galaxeye_results/
!cp -r results/ /content/drive/MyDrive/galaxeye_results/
!cp config.yaml /content/drive/MyDrive/galaxeye_results/
!cp norm_stats.json /content/drive/MyDrive/galaxeye_results/

print("\n✓ All results backed up to Google Drive → galaxeye_results/")



✓ All results backed up to Google Drive → galaxeye_results/


# CELL 14 — View Results

In [ ]:
import json
with open("results/metrics_test.json") as f:
    results = json.load(f)

print("\n" + "="*60)
print("  TEST SET RESULTS — USE THESE IN YOUR REPORT")
print("="*60)
print(f"  IoU       : {results['iou']}")
print(f"  Precision : {results['precision']}")
print(f"  Recall    : {results['recall']}")
print(f"  F1 Score  : {results['f1']}")
print(f"  Threshold : {results['threshold']}")
print("="*60)
print("\nConfusion Matrix:")
cm = np.array(results['confusion_matrix'])
print(f"  [[TN={cm[0,0]:>6,}   FP={cm[0,1]:>6,}]")
print(f"   [FN={cm[1,0]:>6,}   TP={cm[1,1]:>6,}]]")
print("\nAll outputs saved to:")
print("  • checkpoints/best_model.pth")
print("  • results/metrics_test.json")
print("  • results/training_history.json")
print("  • Google Drive → galaxeye_results/")



FileNotFoundError: [Errno 2] No such file or directory: 'results/metrics_test.json'

# CELL 15 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

with open("results/training_history.json") as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1"], color="green")
axes[1].set_title("Validation F1"); axes[1].set_xlabel("Epoch")

axes[2].plot(history["val_iou"], color="purple")
axes[2].set_title("Validation IoU"); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=120)
plt.show()

print("✓ Training curves saved to results/training_curves.png")




# END OF NOTEBOOK

In [ ]:
import torch
print(torch.cuda.is_available()) # Should return True

False
